# Options from Zero · Episode 2: Maths pit stop: exp, log and continuous compounding

Companion notebook for the video. Every number that appears on screen or in the narration is produced by running this notebook and read from `build/outputs.json`.

> **The running trade.** An illustrative share priced at \$100 that pays no dividends, a one-year call option on it with a strike of \$105, 20% volatility and a 3.60% interest rate (continuously compounded). All made up for teaching; it is not market data.
>
> Educational material only, not financial advice. Options are risky: a seller can lose far more than the premium received.

1. Setup · 2. Last episode's question · 3. Where e comes from · 4. Our rate, compounded more often · 5. Discounting · 6. Logs and returns · 7. Where they appear next · 8. Export

**Running in Google Colab?** Run the next cells first: they install QuantLib (version 1.43, the one used in the video) and write the data files this notebook reads. Then run the rest of the notebook in order.

In [ ]:
# Colab doesn't include QuantLib. Install it (about 30 seconds).
# The video used QuantLib 1.43; drop '==1.43' for the latest.
!pip install QuantLib==1.43

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"

## 1. Setup

In [ ]:
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import QuantLib as ql

out = {"meta": {
    "series": "Options from Zero",
    "episode": 2,
    "quantlib_version": ql.__version__,
    "quotes_label": "illustrative (made up for teaching)",
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}}
S, K, T, r, vol = 100.0, 105.0, 1.0, 0.036, 0.20   # the running trade

The option formulas used throughout the series (`episodes/_shared/options.py`):

In [ ]:
# Option formulas for Options from Zero, written out by hand so every step can be read. Copied verbatim into
# each notebook by make_notebook.py, so every notebook runs on its own; the notebooks check these against
# QuantLib. Needs: math, numpy as np.
#   Running trade (Part 1): an illustrative share at $100, no dividends, 20% volatility, a one-year call struck
#   at $105, interest 3.60% a year continuously compounded. Made up for teaching.

def norm_cdf(x):
    """N(x): the chance a standard normal variable is below x."""
    return 0.5 * (1 + math.erf(x / math.sqrt(2)))

def call_payoff(s, k):
    return np.maximum(np.asarray(s, dtype=float) - k, 0.0)

def put_payoff(s, k):
    return np.maximum(k - np.asarray(s, dtype=float), 0.0)

def black_scholes(s, k, t, r, vol, call=True):
    """Black-Scholes price of a European option on a share with no dividends (r continuously compounded)."""
    d1 = (math.log(s / k) + (r + vol ** 2 / 2) * t) / (vol * math.sqrt(t))
    d2 = d1 - vol * math.sqrt(t)
    if call:
        return s * norm_cdf(d1) - k * math.exp(-r * t) * norm_cdf(d2)
    return k * math.exp(-r * t) * norm_cdf(-d2) - s * norm_cdf(-d1)

def one_step(s, up, down, k, t, r, call=True):
    """One-step tree: replicate the option with delta shares and cash. Returns the pieces and the price."""
    pay = call_payoff if call else put_payoff
    v_up, v_down = float(pay(up, k)), float(pay(down, k))
    delta = (v_up - v_down) / (up - down)
    cash = (v_down - delta * down) * math.exp(-r * t)   # negative: borrowed today
    q = (s * math.exp(r * t) - down) / (up - down)       # risk-neutral probability of the up move
    return {"v_up": v_up, "v_down": v_down, "delta": delta, "cash": cash, "price": delta * s + cash, "q": q}

def binomial_tree(s, k, t, r, vol, steps, call=True, american=False, quantlib_p=False):
    """Cox-Ross-Rubinstein tree: u = exp(vol sqrt(dt)), d = 1/u, worked backward from expiry.
    quantlib_p=True uses QuantLib's up-probability 1/2 + 1/2 (r - vol^2/2) sqrt(dt) / vol instead of the textbook
    (e^{r dt} - d) / (u - d); both give the same price in the limit."""
    dt = t / steps
    u = math.exp(vol * math.sqrt(dt)); d = 1 / u
    p = 0.5 + 0.5 * (r - vol ** 2 / 2) * math.sqrt(dt) / vol if quantlib_p else (math.exp(r * dt) - d) / (u - d)
    pay = call_payoff if call else put_payoff
    prices = s * d ** np.arange(steps, -1, -1) * u ** np.arange(steps + 1)
    values = pay(prices, k)
    for _ in range(steps):
        prices = prices[1:] / u
        values = math.exp(-r * dt) * (p * values[1:] + (1 - p) * values[:-1])
        if american:
            values = np.maximum(values, pay(prices, k))
    return float(values[0])

## 2. Last episode's question

You sold the call for its premium and the share ends at \$130: you owe the buyer the payoff, \$130 − \$105.

In [ ]:
premium = black_scholes(S, K, T, r, vol)
out["answer"] = {"premium": premium, "payoff": 130.0 - K, "profit": premium - (130.0 - K)}
out["answer"]

## 3. Where e comes from

Put \$1 in an account paying 100% a year. Paid once, you end with \$2. Paid twice (50% each half), \$2.25. The more often interest is paid, the more you get, but it never passes one number: $e = 2.71828\ldots$

In [ ]:
ns = [1, 2, 4, 12, 365, 1_000_000]
growth = {n: (1 + 1 / n) ** n for n in ns}
out["e"] = {"rows": [{"times": n, "grows_to": g} for n, g in growth.items()], "e": math.e,
            "gap_daily": math.e - growth[365]}
assert abs(growth[1_000_000] - math.e) < 2e-6
pd.DataFrame(out["e"]["rows"])

## 4. Our rate, compounded more often

The running trade's 3.60% works the same way. Compounded continuously, \$1 grows to $e^{rT}$ in $T$ years.

In [ ]:
labels = {1: "once a year", 4: "quarterly", 12: "monthly", 365: "daily"}
rows = [{"how": labels[n], "grows_to": (1 + r / n) ** n} for n in labels]
rows.append({"how": "continuously", "grows_to": math.exp(r * T)})
out["compound"] = {"rate_pct": round(r * 100, 6), "rows": rows, "annual": rows[0]["grows_to"],
                   "continuous": math.exp(r * T), "forward_100": S * math.exp(r * T),
                   "extra_cents_per_100": (math.exp(r) - (1 + r)) * 100}
pd.DataFrame(rows)

## 5. Discounting

Running it backwards: a dollar paid in $T$ years is worth $e^{-rT}$ today. This discount factor appears in every option price in the series.

In [ ]:
df = math.exp(-r * T)
out["discount"] = {"df": df, "cents": df * 100, "pv_of_103_67": S * math.exp(r * T) * df, "future": S * math.exp(r * T)}
out["discount"]

## 6. Logs and returns

The natural log undoes exp: $\ln(e^{x}) = x$. It turns growth *factors* into *rates* that add up. A share that rises 10% and then falls 10% ends 1% down: the percentages don't add to zero, but the log returns add to exactly the log of the total change.

In [ ]:
up, down = 1.10, 0.90
out["logs"] = {"up_pct": 10, "down_pct": 10, "total_pct_abs": abs((up * down - 1) * 100), "ln_growth": math.log(math.exp(r * T)), "total_factor": up * down, "total_pct": (up * down - 1) * 100,
               "log_up": math.log(up), "log_down": math.log(down), "log_sum": math.log(up) + math.log(down),
               "ln_total": math.log(up * down)}
assert abs(out["logs"]["log_sum"] - out["logs"]["ln_total"]) < 1e-15
out["logs"]

## 7. Where they appear next

$e^{rT}$ gives the forward price (Episode 4) and $e^{-rT}$ discounts every payoff; $\ln(S/K)$ sits inside the Black-Scholes formula (Episode 10); log returns measure volatility (Episode 9). Check with QuantLib: its flat 3.60% continuously compounded curve gives the same one-year discount factor.

In [ ]:
today = ql.Date(29, 9, 2026)
curve = ql.FlatForward(today, r, ql.Actual365Fixed(), ql.Continuous)
out["quantlib"] = {"df": curve.discount(today + 365), "diff": abs(curve.discount(today + 365) - df)}
assert out["quantlib"]["diff"] < 1e-15
out["quantlib"]

## 8. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")